In [23]:
import numpy as np
import os
import re

In [37]:
region_map = {'UKNegatives': 'UK',
              'USA': 'USA',
              'Russia': 'Russia',
              'Russia2': 'Russia',
              'Russia3': 'Russia',
              'Karoo': 'Karoo',
              'Brazil': 'Brazil',
              'Australia': 'Australia',
              'France': 'France',
              'Texas': 'USA',
              'UKTraining': 'Somerset',
              'EastQuantock': 'Somerset',
              'UKQuantock': 'Somerset',}

In [11]:
output_dir = '/Users/willmcmahon/Documents/PhD/scd-detection/data/PostIRP/dataset/'

In [ ]:
pattern = r'g\d{4}_'

for dir in ['/Users/willmcmahon/Downloads/Tiles 2', '/Users/willmcmahon/Downloads/Tiles 3']:
    for root, dirs, files in os.walk(dir):
        for file in files:
            if file.endswith('.npz'):
                data = np.load(os.path.join(root, file))
                resolution = data['res']
                positive = 0 if data['scd_pixel_fraction'] == 0 else 1
                positive_str = 'positive' if positive == 1 else 'negative'
                region = os.path.basename(os.path.dirname(os.path.join(root, file)))  # parent directory name of the specific file
                region = region_map[region]

                if 'Copy of' in file:
                    file = file.replace('Copy of ', '')

                file = file.replace('pos_', '')
                file = file.replace('neg_', '')

                # find and remove group indicator (g0001, g0002, etc.)
                file = re.sub(pattern, '', file)

                bounds = file.strip('.npz').split('_')[-4:] # last 4 in path are bounds
                for i in range(len(bounds)):
                    try:
                        bounds[i] = int(bounds[i])
                    except ValueError:
                        print(f"Could not convert {bounds[i]} to int for file {file}")


                name = f"{region}_{positive_str}_{file}"

                np.savez_compressed(os.path.join(output_dir, name),
                        image=data['image'], 
                        layer_names=data['layer_names'],
                        res=resolution,
                        labels = data['labels'],
                        scd_pixel_fraction=data['scd_pixel_fraction'],
                        positive=positive,
                        region=region,
                        bounds = tuple(bounds))
            